In [0]:
%sql
-- Create silver schema
CREATE SCHEMA IF NOT EXISTS caracare_casestudy.`01_silver`
COMMENT 'Silver layer - cleaned and validated data';

In [0]:
%sql
-- Clean and load dim_patient to silver
CREATE OR REPLACE TABLE caracare_casestudy.`01_silver`.dim_patient
USING DELTA
COMMENT 'Silver layer - Patient dimension with cleaned and validated data'
AS
SELECT DISTINCT
    patient_id,
    TRIM(LOWER(patient_email)) AS patient_email,  -- Standardize email format
    date_of_birth
FROM caracare_casestudy.`00_bronze`.dim_patient
WHERE patient_id IS NOT NULL  -- Remove null primary keys
  AND date_of_birth IS NOT NULL  -- Ensure valid birth dates
  AND patient_email IS NOT NULL;  -- Require email

In [0]:
%sql
-- Clean and load dim_doctor to silver
CREATE OR REPLACE TABLE caracare_casestudy.`01_silver`.dim_doctor
USING DELTA
COMMENT 'Silver layer - Doctor dimension with cleaned and validated data'
AS
SELECT DISTINCT
    doctor_id,
    TRIM(prescribing_doctor) AS prescribing_doctor,  -- Remove leading/trailing spaces
    TRIM(doctor_specialty) AS doctor_specialty,
    TRIM(UPPER(doctor_city)) AS doctor_city  -- Standardize city names to uppercase
FROM caracare_casestudy.`00_bronze`.dim_doctor
WHERE doctor_id IS NOT NULL  -- Remove null primary keys
  AND prescribing_doctor IS NOT NULL;  -- Require doctor name

In [0]:
%sql
-- Clean and load dim_insurance to silver
CREATE OR REPLACE TABLE caracare_casestudy.`01_silver`.dim_insurance
USING DELTA
COMMENT 'Silver layer - Insurance dimension with cleaned and validated data'
AS
SELECT DISTINCT
    insurance_id,
    TRIM(insurance_name) AS insurance_name,  -- Remove leading/trailing spaces
    TRIM(UPPER(insurance_type)) AS insurance_type  -- Standardize type to uppercase
FROM caracare_casestudy.`00_bronze`.dim_insurance
WHERE insurance_id IS NOT NULL  -- Remove null primary keys
  AND insurance_name IS NOT NULL;  -- Require insurance name

In [0]:
%sql
-- Clean and load fact_prescription to silver
CREATE OR REPLACE TABLE caracare_casestudy.`01_silver`.fact_prescription
USING DELTA
COMMENT 'Silver layer - Prescription fact with cleaned and validated data'
AS
SELECT DISTINCT
    prescription_id,
    patient_id,
    doctor_id,
    insurance_id,
    prescription_start,
    prescription_end,
    TRIM(UPPER(prescription_status)) AS prescription_status,  -- Standardize status
    represcription,
    represcription_date
FROM caracare_casestudy.`00_bronze`.fact_prescription
WHERE prescription_id IS NOT NULL  -- Remove null primary keys
  AND patient_id IS NOT NULL  -- Require valid foreign key
  AND doctor_id IS NOT NULL  -- Require valid foreign key
  AND insurance_id IS NOT NULL  -- Require valid foreign key
  AND prescription_start IS NOT NULL  -- Require start date
  AND prescription_start <= COALESCE(prescription_end, prescription_start)  -- Validate date range

In [0]:
%sql
-- Clean and load fact_touchpoint to silver
CREATE OR REPLACE TABLE caracare_casestudy.`01_silver`.fact_touchpoint
USING DELTA
COMMENT 'Silver layer - Touchpoint fact with cleaned and validated data'
AS
SELECT DISTINCT
    touchpoint_id,
    prescription_id,
    touchpoint_date,
    TRIM(UPPER(touchpoint_type)) AS touchpoint_type,  -- Standardize type
    TRIM(UPPER(touchpoint_channel)) AS touchpoint_channel,  -- Standardize channel
    TRIM(touchpoint_outcome) AS touchpoint_outcome  -- Remove spaces
FROM caracare_casestudy.`00_bronze`.fact_touchpoint
WHERE touchpoint_id IS NOT NULL  -- Remove null primary keys
  AND prescription_id IS NOT NULL  -- Require valid foreign key
  AND touchpoint_date IS NOT NULL;  -- Require touchpoint date

In [0]:
%sql
-- Validation: Check row counts for all silver tables
SELECT 
    'dim_patient' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`01_silver`.dim_patient

UNION ALL

SELECT 
    'dim_doctor' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`01_silver`.dim_doctor

UNION ALL

SELECT 
    'dim_insurance' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`01_silver`.dim_insurance

UNION ALL

SELECT 
    'fact_prescription' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`01_silver`.fact_prescription

UNION ALL

SELECT 
    'fact_touchpoint' AS table_name,
    COUNT(*) AS row_count
FROM caracare_casestudy.`01_silver`.fact_touchpoint

ORDER BY table_name;